In [1]:

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import tensorflow as tf

from src.catalog_classifier import (
    download_fashion_product_images,
    fashion_product_datasets,
)

from src.transfer_learning import build_resnet50v2_classifier

tf.keras.utils.set_random_seed(42)

print("TensorFlow:", tf.__version__)


TensorFlow: 2.21.0


In [2]:

dataset_dir = download_fashion_product_images()

train_ds, validation_ds, class_names, dataset_summary = fashion_product_datasets(
    dataset_dir,
    label_column="subCategory",
    image_size=(128, 128),
    batch_size=16,
    min_examples_per_class=100,
    seed=42,
)

print(dataset_summary)
print("Categories:", class_names)


{'label_column': 'subCategory', 'source_rows': 43996, 'train_rows': 35186, 'validation_rows': 8810, 'class_counts': {'Accessories': 143, 'Apparel Set': 106, 'Bags': 3055, 'Belts': 811, 'Bottomwear': 2693, 'Cufflinks': 108, 'Dress': 478, 'Eyewear': 1073, 'Flip Flops': 915, 'Fragrance': 1012, 'Free Gifts': 104, 'Headwear': 293, 'Innerwear': 1808, 'Jewellery': 1080, 'Lips': 527, 'Loungewear and Nightwear': 470, 'Makeup': 307, 'Nails': 329, 'Sandal': 963, 'Saree': 427, 'Scarves': 118, 'Shoes': 7344, 'Socks': 698, 'Ties': 258, 'Topwear': 15401, 'Wallets': 933, 'Watches': 2542}}
Categories: ['Accessories', 'Apparel Set', 'Bags', 'Belts', 'Bottomwear', 'Cufflinks', 'Dress', 'Eyewear', 'Flip Flops', 'Fragrance', 'Free Gifts', 'Headwear', 'Innerwear', 'Jewellery', 'Lips', 'Loungewear and Nightwear', 'Makeup', 'Nails', 'Sandal', 'Saree', 'Scarves', 'Shoes', 'Socks', 'Ties', 'Topwear', 'Wallets', 'Watches']


In [3]:

resnet_model, resnet_base = build_resnet50v2_classifier(
    input_shape=(128, 128, 3),
    class_count=len(class_names),
)

resnet_model.summary()


94668760/94668760 ━━━━━━━━━━━━━━━━━━━━ 11s 0us/step


Model: "retailmind_resnet50v2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50v2 (Functional)         │ (None, 4, 4, 2048)     │    23,564,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 27)             │        55,323 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,620,123 (90.10 MB)

 Trainable params: 55,323 (216.11 KB)

 Non-trainable params: 23,564,800 (89.89 MB)

In [5]:

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

frozen_history = resnet_model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=5,
    callbacks=[early_stopping],
)

frozen_loss, frozen_accuracy = resnet_model.evaluate(
    validation_ds,
    verbose=0,
)

print(f"Frozen Backbone Accuracy: {frozen_accuracy:.4f}")


Epoch 1/5
2200/2200 ━━━━━━━━━━━━━━━━━━━━ 1230s 558ms/step - accuracy: 0.8676 - loss: 0.5112 - val_accuracy: 0.9085 - val_loss: 0.3642
Epoch 2/5
2200/2200 ━━━━━━━━━━━━━━━━━━━━ 1076s 489ms/step - accuracy: 0.8861 - loss: 0.4412 - val_accuracy: 0.9137 - val_loss: 0.3417
Epoch 3/5
2200/2200 ━━━━━━━━━━━━━━━━━━━━ 1067s 485ms/step - accuracy: 0.8939 - loss: 0.4176 - val_accuracy: 0.9194 - val_loss: 0.3491
Epoch 4/5
2200/2200 ━━━━━━━━━━━━━━━━━━━━ 1075s 488ms/step - accuracy: 0.8932 - loss: 0.4221 - val_accuracy: 0.9182 - val_loss: 0.3509
Epoch 5/5
2200/2200 ━━━━━━━━━━━━━━━━━━━━ 1098s 499ms/step - accuracy: 0.8987 - loss: 0.4081 - val_accuracy: 0.9204 - val_loss: 0.3744
Frozen Backbone Accuracy: 0.9137


In [6]:

resnet_base.trainable = True

for layer in resnet_base.layers[:-30]:
    layer.trainable = False

for layer in resnet_base.layers[-30:]:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

resnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

print("Trainable layers:", sum(
    layer.trainable for layer in resnet_base.layers
))


Trainable layers: 22


In [8]:

fine_tuning_early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

fine_tuning_history = resnet_model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=5,
    callbacks=[fine_tuning_early_stopping],
)

fine_tuned_loss, fine_tuned_accuracy = resnet_model.evaluate(
    validation_ds,
    verbose=0,
)

print(f"Fine-tuned Accuracy: {fine_tuned_accuracy:.4f}")


Epoch 1/5
2200/2200 ━━━━━━━━━━━━━━━━━━━━ 1948s 885ms/step - accuracy: 0.9265 - loss: 0.2405 - val_accuracy: 0.9435 - val_loss: 0.1987
Epoch 2/5
2200/2200 ━━━━━━━━━━━━━━━━━━━━ 1933s 878ms/step - accuracy: 0.9443 - loss: 0.1794 - val_accuracy: 0.9524 - val_loss: 0.1727
Epoch 3/5
2200/2200 ━━━━━━━━━━━━━━━━━━━━ 1951s 887ms/step - accuracy: 0.9550 - loss: 0.1435 - val_accuracy: 0.9510 - val_loss: 0.1704
Epoch 4/5
2200/2200 ━━━━━━━━━━━━━━━━━━━━ 1944s 884ms/step - accuracy: 0.9606 - loss: 0.1210 - val_accuracy: 0.9569 - val_loss: 0.1577
Epoch 5/5
2200/2200 ━━━━━━━━━━━━━━━━━━━━ 1935s 879ms/step - accuracy: 0.9659 - loss: 0.1056 - val_accuracy: 0.9604 - val_loss: 0.1536
Fine-tuned Accuracy: 0.9604


In [9]:

import json

day6_metrics_path = PROJECT_ROOT / "models" / "catalog_cnn_day6_metrics.json"

with day6_metrics_path.open(encoding="utf-8") as file:
    day6_metrics = json.load(file)

day6_accuracy = day6_metrics["validation_accuracy"]

print(f"Day 6 CNN:              {day6_accuracy:.4%}")
print(f"Day 7 Frozen ResNet:    {frozen_accuracy:.4%}")
print(f"Day 7 Fine-tuned ResNet: {fine_tuned_accuracy:.4%}")

print(
    f"\nImprovement over Day 6: "
    f"{(fine_tuned_accuracy - day6_accuracy) * 100:+.2f} percentage points"
)


Day 6 CNN:              84.9262%
Day 7 Frozen ResNet:    91.3734%
Day 7 Fine-tuned ResNet: 96.0386%

Improvement over Day 6: +11.11 percentage points


In [10]:

from src.catalog_classifier import save_catalog_classifier

model_path = PROJECT_ROOT / "models" / "catalog_resnet50v2_day7.keras"

save_catalog_classifier(
    resnet_model,
    class_names,
    model_path,
)

day7_metrics = {
    "architecture": "ResNet50V2",
    "pretrained_weights": "imagenet",
    "frozen_backbone_accuracy": float(frozen_accuracy),
    "fine_tuned_accuracy": float(fine_tuned_accuracy),
    "day6_cnn_accuracy": float(day6_accuracy),
    "improvement_over_day6": float(fine_tuned_accuracy - day6_accuracy),
    **dataset_summary,
}

metrics_path = model_path.with_name(
    "catalog_resnet50v2_day7_metrics.json"
)

metrics_path.write_text(
    json.dumps(day7_metrics, indent=2),
    encoding="utf-8",
)

print("Model saved:", model_path)
print("Metrics saved:", metrics_path)


Model saved: C:\Users\Abhishek\Desktop\RetailMind-AI\models\catalog_resnet50v2_day7.keras
Metrics saved: C:\Users\Abhishek\Desktop\RetailMind-AI\models\catalog_resnet50v2_day7_metrics.json
